In [32]:
import cooler
import numpy as np
import pandas as pd

In [33]:
### to load a cooler with a specific resolution use the following syntax:
clr_1 = cooler.Cooler("/usr/users/papantonis1/aman/microc_data/nadine_macro/RBP1-MNase-R1-filtered.mcool::resolutions/10000")
clr_2 = cooler.Cooler("/usr/users/papantonis1/aman/microc_data/nadine_macro/ctrl-MNase-R1-filtered.mcool::resolutions/10000")

In [34]:
# Testing only chr. 14
#binsize = clr_1.binsize
chrom = "chr14"
matrix = clr_1.matrix(balance=True).fetch(chrom)
bins = clr_1.bins().fetch(chrom)

In [35]:
bins.tail()

,chrom,start,end,weight
229846,chr14,107000000,107010000,NaN
229847,chr14,107010000,107020000,NaN
229848,chr14,107020000,107030000,NaN
229849,chr14,107030000,107040000,NaN
229850,chr14,107040000,107043718,NaN


In [36]:
from scipy.ndimage import uniform_filter
masked = np.isnan(matrix)
matrix_filled = np.where(masked, 0, matrix)
smoothed = uniform_filter(matrix_filled, size=7, mode='constant')
print(np.count_nonzero(~np.isnan(smoothed)))
#binsize

# checking to see if it only has zeo values
print(np.min(smoothed), np.max(smoothed), np.mean(smoothed))
smoothed = np.log1p(smoothed)
smoothed.shape # = 2 bins i,j

114597025
-4.907176475681713e-17 0.16517374525448297 9.706888996539097e-05


(10705, 10705)

In [37]:
bin_midpoints = ((bins['start'] + bins['end']) // 2).values  # ← `.values` gives numpy array ## floor div

In [38]:
bin_midpoints # 10kb bin

array([     5000,     15000,     25000, ..., 107025000, 107035000,
       107041859], dtype=int32)

In [39]:
genomic_distances = np.abs(bin_midpoints[:, None] - bin_midpoints)

In [40]:
smoothed = np.clip(smoothed, a_min=0, a_max=None)

In [41]:
dist_flat = genomic_distances.flatten()
signal_flat = smoothed.flatten()

In [42]:
signal_flat

array([0., 0., 0., ..., 0., 0., 0.])

In [43]:
print(np.count_nonzero(signal_flat))

64967081


In [44]:
df = pd.DataFrame({'distance': dist_flat, 'signal': signal_flat}).dropna()

In [45]:
(df['signal'] != 0).sum()

64967081

### on to grouping now

In [46]:
resolution = 10000
df['distance_group'] = (df['distance'] // resolution) * resolution

grouped = df.groupby('distance_group')['signal']
q1 = grouped.quantile(0.25)
q3 = grouped.quantile(0.75)

In [47]:
#grouped.head()
#q1
#q3

In [48]:
df

,distance,signal,distance_group
0,0,0.0,0
1,10000,0.0,10000
2,20000,0.0,20000
3,30000,0.0,30000
4,40000,0.0,40000
...,...,...,...
114597020,36859,0.0,30000
114597021,26859,0.0,20000
114597022,16859,0.0,10000
114597023,6859,0.0,0


In [49]:
# merge back
df = df.join(q1, on='distance_group', rsuffix='_q1')
df = df.join(q3, on='distance_group', rsuffix='_q3')

In [50]:
df

,distance,signal,distance_group,signal_q1,signal_q3
0,0,0.0,0,0.057683,0.079332
1,10000,0.0,10000,0.053630,0.073394
2,20000,0.0,20000,0.047015,0.064274
3,30000,0.0,30000,0.039541,0.054053
4,40000,0.0,40000,0.031631,0.043329
...,...,...,...,...,...
114597020,36859,0.0,30000,0.039541,0.054053
114597021,26859,0.0,20000,0.047015,0.064274
114597022,16859,0.0,10000,0.053630,0.073394
114597023,6859,0.0,0,0.057683,0.079332


In [51]:
df['iqr'] = df['signal_q3'] - df['signal_q1']
df = df[df['iqr'] != 0]
df['fold_change_iqr'] = (df['signal'] - df['signal_q3']) / df['iqr']
df['fold_change_iqr']

/tmp/ipykernel_3206080/2822709293.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['fold_change_iqr'] = (df['signal'] - df['signal_q3']) / df['iqr']


0           -3.664420
1           -3.713536
2           -3.724101
3           -3.724672
4           -3.704043
               ...   
114597020   -3.724672
114597021   -3.724101
114597022   -3.713536
114597023   -3.664420
114597024   -3.664420
Name: fold_change_iqr, Length: 103650311, dtype: float64

In [52]:
# filter outlier
#outliers = df[df['fold_change_iqr'] > 10000]
#len(outliers)
#outliers

In [53]:
#outliers

In [54]:
# change to log fold
df.loc[:, 'log_fc'] = np.log10(df['fold_change_iqr'].replace(0, np.nan))

/usr/users/papantonis1/anaconda3/envs/cool_notebook/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipykernel_3206080/3360274830.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:, 'log_fc'] = np.log10(df['fold_change_iqr'].replace(0, np.nan))


In [55]:
outliers = df[(df['log_fc'] > 3) & (df['signal'] > 1e-4)]

In [56]:
#outliers
len(outliers)

44

In [57]:
outliers = outliers[outliers['distance'] > 1000000]

In [58]:
len(outliers)

44

In [59]:
outliers

,distance,signal,distance_group,signal_q1,signal_q3,iqr,fold_change_iqr,log_fc
21546311,58390000,0.000101,58390000,0.0,8.571628e-18,8.571628e-18,1.181479e+13,13.072426
21546313,58410000,0.000103,58410000,0.0,8.413423e-18,8.413423e-18,1.228941e+13,13.089531
21556983,58050000,0.000107,58050000,0.0,1.036789e-17,1.036789e-17,1.035940e+13,13.015335
21899864,60940000,0.000103,60940000,0.0,5.079224e-18,5.079224e-18,2.025878e+13,13.306613
21910568,60920000,0.000103,60920000,0.0,5.172018e-18,5.172018e-18,1.990071e+13,13.298869
21910569,60930000,0.000127,60930000,0.0,5.080192e-18,5.080192e-18,2.503788e+13,13.398598
21921273,60910000,0.000103,60910000,0.0,5.297863e-18,5.297863e-18,1.942799e+13,13.288428
21921274,60920000,0.000127,60920000,0.0,5.172018e-18,5.172018e-18,2.459335e+13,13.390818
21921275,60930000,0.000107,60930000,0.0,5.080192e-18,5.080192e-18,2.096895e+13,13.321577
21921276,60940000,0.000107,60940000,0.0,5.079224e-18,5.079224e-18,2.097295e+13,13.321659


In [62]:
n_bins = smoothed.shape[0]
outliers['bin1'] = outliers.index // n_bins
outliers['bin2'] = outliers.index % n_bins

outliers['chrom1'] = chrom
outliers['start1'] = bins.iloc[outliers['bin1']]['start'].values
outliers['end1']   = bins.iloc[outliers['bin1']]['end'].values

outliers['chrom2'] = chrom
outliers['start2'] = bins.iloc[outliers['bin2']]['start'].values
outliers['end2']   = bins.iloc[outliers['bin2']]['end'].values

# Select columns for BEDPE
bedpe_cols = ['chrom1', 'start1', 'end1', 'chrom2', 'start2', 'end2', 'signal', 'fold_change_iqr']
outliers[bedpe_cols].to_csv("clod_chr14_outliers_filtered.bedpe", sep='\t', index=False)